# Semana 06 — Limpeza e Transformação de Dados

**Curso:** Análise de Dados com Python — SENAI (Turma T5)
**UC (MSEP):** Manipulação de Dados com Python e SQL (150h) — Bloco 3, Semanas 05 e 06 (fecha o bloco)

Esta semana usa um cenário novo: o **Restaurante Sabor Caseiro**, que tem 2 filiais (Centro e Zona Sul), e o **DRE (Demonstrativo de Resultado do Exercício)** de cada uma — o relatório financeiro que resume receitas e despesas de um período. Os dois relatórios chegaram direto de um sistema novo, sem nenhuma limpeza: 50 linhas ao todo, com 9 valores ausentes, linhas duplicadas e um erro de digitação em cada filial. Você vai aprender a detectar e corrigir cada um desses problemas — e, no final, vai gerar o arquivo consolidado que o time de BI vai usar para montar os painéis do Power BI.

> Em cada tópico abaixo: **exemplos resolvidos** + **atividade prática** para você fazer sozinho(a).

### 🧭 De onde você vem: fechando a Semana 05

Antes de entrar na Semana 06, vale reforçar de onde você está vindo e pra onde está indo dentro do curso — e relembrar rapidinho o que cada ferramenta que você usou faz.

**O que você viu e estudou na Semana 05:** a dupla Pandas + NumPy. Você criou **arrays NumPy** — uma estrutura parecida com uma lista, mas otimizada pra fazer conta em cima de todos os números de uma vez (por exemplo, `array * 2` multiplica cada elemento do array, sem precisar de um `for` percorrendo item por item). Depois leu um arquivo real com **`pd.read_csv()`** — que abre um arquivo CSV e já devolve ele pronto em formato de tabela (um DataFrame) — usando o `vendas_papelaria.csv`, da Papelaria Boa Ideia. Em cima desse DataFrame, você usou **`describe()`** (calcula estatísticas resumidas — média, desvio padrão, quartis — de cada coluna numérica, pra ter uma primeira noção dos dados antes de qualquer gráfico), **`.loc`/`.iloc`** (selecionam linhas e colunas por nome ou por posição, quando você quer só um pedaço específico da tabela, não ela inteira) e **`groupby()`** (agrupa as linhas por uma coluna categórica — por exemplo, categoria do produto — e permite somar ou tirar a média dentro de cada grupo, o mesmo papel de uma tabela dinâmica do Excel).

**O que você fez:** cada exemplo e atividade daquela semana usou esse mesmo arquivo real (nunca dado digitado à mão) e essas mesmas ferramentas — `read_csv()` pra abrir, `describe()`/`.loc`/`.iloc`/`groupby()` pra explorar. Na sexta-feira, seu squad recebeu um cenário de negócio pra praticar essa mesma extração de informação.

**Qual era o objetivo:** te dar as ferramentas pra abrir qualquer arquivo real e tirar dele uma resposta — quantidade, média, agrupamento por categoria — sem escrever um `for` pra cada conta.

**Onde você está agora:** no meio do Bloco 3 desta UC ("Pandas/NumPy — você extrai e limpa informação de dados reais"). A Semana 05 resolveu a parte de **extrair**; a Semana 06, que começa agora, fecha esse bloco resolvendo a parte de **limpar** — porque, na prática, quase nenhum dado real chega pronto pra virar gráfico ou decisão.

---
### 🔄 Retomada — o que vimos na Semana 05

Você já sabe criar arrays NumPy, ler um CSV real com `pd.read_csv()`, usar `describe()`, `.loc`/`.iloc`, filtrar linhas, agrupar com `groupby()` e confirmar que uma coluna do Pandas é, por dentro, um array NumPy. Essa semana usa tudo isso — e adiciona o que fazer quando o dataset que você lê **não está limpo**.

### 🟢 Abertura — Semana 06: Limpeza e Transformação de Dados

Dados reais nunca chegam perfeitos. Estimativas de mercado apontam que **60 a 80% do tempo de um analista de dados júnior** vai para preparar os dados antes de qualquer gráfico ou modelo — não para a análise em si. Nesta semana você aprende exatamente esse trabalho, usando relatórios financeiros reais.

**O que você vai aprender nesta semana:**
- Detectar e tratar valores ausentes (NaN) com `isna()`, `dropna()` e `fillna()`
- Detectar e remover duplicidades e tratar outliers com `duplicated()` e IQR
- Normalizar dados e criar colunas calculadas/condicionais
- Limpar texto sujo (espaço sobrando, letra errada, palavra quebrada) e converter coluna de dinheiro digitada como texto
- Padronizar nome de coluna e substituir valores específicos dentro de uma coluna
- Combinar DataFrames com `pd.concat()`, montar um pipeline de limpeza completo e **exportar o resultado para o time de BI**

### 📌 Antes de começar: o dataset desta semana

O **DRE (Demonstrativo de Resultado do Exercício)** é o relatório contábil que resume, período a período, quanto uma empresa recebeu (receitas) e gastou (despesas). O Restaurante Sabor Caseiro tem 2 filiais, e cada uma manda seu próprio relatório mensal:

- `dataset/dre_filial_centro.csv` — 25 linhas (Jan a Abr, 6 categorias por mês, mais 1 linha duplicada)
- `dataset/dre_filial_zona_sul.csv` — 25 linhas (mesma estrutura)

Juntas, as duas filiais somam **50 linhas** e **9 valores ausentes** no total — nenhuma limpeza foi feita ainda. As 6 categorias são as mesmas em toda filial e todo mês: `Receita de Vendas`, `Impostos sobre Vendas`, `CMV`, `Despesas com Pessoal`, `Despesas Administrativas`, `Despesas Financeiras`.

**Rodando no Google Colab?** Diferente do VS Code local, o Colab não enxerga os arquivos desta pasta automaticamente — rode a célula abaixo pra enviar os 2 arquivos; ela já coloca tudo na subpasta `dataset/` certinha. Os datasets dos squads (seção 6) só precisam ser enviados na sexta-feira, quando cada squad for trabalhar com o seu.

In [ ]:
# Se estiver rodando no Google Colab, esta célula abre a janela de upload
# e organiza os arquivos na subpasta dataset/, igual à estrutura do repositório.
# Selecione os 2 arquivos desta semana: dre_filial_centro.csv e dre_filial_zona_sul.csv
import os

try:
    from google.colab import files
    enviados = files.upload()
    os.makedirs("dataset", exist_ok=True)
    for nome in enviados:
        os.replace(nome, f"dataset/{nome}")
except ImportError:
    print("Rodando localmente (VS Code) — os arquivos já estão em dataset/, na pasta desta semana.")

💡 **Lembrando das semanas anteriores:** `os` é o módulo de sistema operacional do Python — já vem pronto, sem precisar instalar. `os.makedirs("dataset", exist_ok=True)` cria a subpasta `dataset/` (sem erro caso ela já exista); `os.replace(nome, f"dataset/{nome}")`, dentro do `for`, move cada arquivo enviado da raiz da sessão pra dentro dessa subpasta.

---
## 1. Valores Ausentes (NaN)

### 🔹 Exemplo 1 — Vendo os dados brutos, sem filtro nenhum

📖 **Antes do código:** antes de detectar qualquer problema com comandos, vale ver os dados exatamente como chegaram — sem filtro, sem tratamento. Vamos trabalhar primeiro com o DRE da filial Centro; a filial Zona Sul entra em cena na Seção 4, quando for hora de consolidar as duas.

📦 **Antes do código: instalando o `pandas` e o `numpy`.** Mesma lógica das semanas anteriores: bibliotecas de terceiros que costumam já vir prontas no Colab e em instalações comuns — mas rodar a célula de instalação garante que funciona em qualquer ambiente, sem custo nenhum se já estiver instalado.

> ⚠️ **Se você pular esta célula** e alguma das duas não estiver instalada, ao rodar o próximo bloco vai aparecer `ModuleNotFoundError`.

In [ ]:
%pip install -q pandas numpy

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")
display(centro)
print(centro.shape)

Olhando esse DRE de 25 linhas, dá pra ver pelo menos 3 problemas diferentes, mesmo sem rodar nenhum comando ainda: várias linhas de `valor` aparecem em branco (`NaN`), a última linha repete exatamente a primeira (`Receita de Vendas`, Janeiro, `45000`), e `Despesas com Pessoal` de Abril está em `125000` — bem mais alto que o esperado pra essa categoria (as outras despesas do relatório ficam na casa dos milhares, não das centenas de milhares). As próximas seções ensinam a encontrar cada um desses problemas com código, sem depender de olhar linha por linha (o que não seria possível com um DRE de milhares de linhas, só com 25).

### 🔹 Exemplo 2 — Conhecendo a estrutura antes de limpar: head(), tail() e info()

📖 **Antes do código:** antes de sair caçando problema por problema, um analista sempre roda um punhado de comandos padrão pra ter uma primeira impressão do dataset. `head()` **mostra as primeiras linhas da tabela** — por padrão, as 5 primeiras (dá pra pedir outro número, ex.: `head(10)`) — e **devolve** um novo DataFrame só com essas linhas; é o comando mais comum pra conferir, logo depois de um `read_csv()`, se o arquivo foi lido do jeito esperado. `tail()` funciona igual a `head()`, só que **com as últimas linhas** (5 por padrão) — também devolve um DataFrame, e costuma ser usado pra checar o final de um arquivo grande, ou pra comparar com o início. `info()` **resume a estrutura inteira da tabela** numa tacada só: quantas linhas e colunas existem, o nome e o tipo de dado (`Dtype`) de cada coluna, e a coluna `Non-Null Count`, que mostra quantos valores **não são nulos** em cada uma — diferente de `head()`/`tail()`, `info()` **já imprime o resumo sozinho** e devolve `None`, por isso ele aparece puro na célula, sem `print(...)` nem `display(...)` em volta (se você envolvesse `centro.info()` num `print(...)`, veria um `None` sobrando no final da saída, já que `print()` também mostraria o valor de retorno). São os 3 comandos que valem a pena rodar toda vez que você abre um arquivo novo, nesta semana e no resto da carreira.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")

display(centro.head())
display(centro.tail())
centro.info()

💡 **Lembrando da Semana 05:** `head()`, `tail()` e `describe()` devolvem um DataFrame — por isso aparecem dentro de `display(...)`, não de `print(...)` (a regra é: `display()` para DataFrame/Series, `print()` para número, texto, array, lista ou dicionário).

Repare que a última linha mostrada por `tail()` (`Receita de Vendas`, Janeiro, `45000`) é idêntica à primeira linha que `head()` mostrou — um indício, só de olhar, de que existe alguma duplicidade. E o `info()` já revela o primeiro problema de verdade: `valor` mostra `20 non-null` num total de `25` linhas — ou seja, `5` valores estão faltando, exatamente o problema que a próxima seção vai aprender a tratar.

📖 **Antes do código: o que `describe()` faz?** `describe()` **calcula estatísticas resumidas** de cada coluna numérica da tabela — `count` (quantos valores não-nulos existem), `mean` (média), `std` (desvio padrão, que mede o quanto os valores de uma coluna se espalham em torno da média; quanto maior o `std`, mais inconsistentes são os números dessa coluna), `min`, `max` e os quartis (`25%`, `50%`, `75%` — você vai ver exatamente o que cada quartil significa na Seção 2, quando forem usados pra encontrar outliers). Ela **devolve** um novo DataFrame com essas estatísticas como linhas. É o comando padrão pra ter uma primeira noção da distribuição de uma coluna numérica, geralmente rodado logo depois de `head()`/`info()`.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")
display(centro.describe())

Aqui, só `valor` é numérica, e `describe()` mostra `count` igual a `20` (as 5 linhas com nulo já ficam de fora da conta automaticamente), `mean` de `22277.50` e um `std` de `30077.65` — bem próximo da própria média, sinal de que tem algo estranho nessa coluna, que a Seção 2 vai investigar de verdade com o método IQR. Por enquanto, repare que esse `describe()` está misturando `Receita de Vendas` com as despesas na mesma conta — a Seção 2 explica por que isso não é ideal, e como corrigir.

### 🔹 Exemplo 3 — Detectando valores ausentes com isna()

📖 **Antes do código:** você já viu, com `info()`, que a coluna `valor` tem 5 valores ausentes — mas isso foi no olho, lendo o resumo. `isna()` faz a mesma detecção de um jeito programático: marca `True`/`False` em **cada célula** da tabela (`True` = valor ausente, `False` = valor preenchido). Encadeado com `.sum()`, que soma esses `True`/`False` **por coluna** (`True` conta como 1, `False` conta como 0), você chega no total de nulos de cada coluna de uma vez — um número que o código consegue usar (por exemplo, dentro de um `if`), em vez de um texto que só o seu olho lê.

In [ ]:
import pandas as pd

centro = pd.read_csv("dataset/dre_filial_centro.csv")
print(centro.isna().sum())

Só `valor` tem valores ausentes — 5 no total, espalhados em meses e categorias diferentes.

📖 **Antes do erro: o que é o `subset` do `dropna()`?** Por padrão, `dropna()` (que você vai usar de verdade daqui a pouco, no Exemplo 4) olha a linha inteira: se qualquer coluna tiver `NaN`, a linha inteira é removida. `subset=["nome_da_coluna"]` restringe essa checagem a uma coluna específica — só remove a linha se *aquela* coluna estiver vazia, ignorando `NaN` que existam em outras colunas. É útil quando só uma coluna realmente importa pra decidir se a linha deve ficar ou sair.

> ⚠️ **Veja como é um erro real do Python.** Mas se o nome dentro de `subset=[...]` estiver errado — `centro.dropna(subset=["valor_errado"])` — você vai ver `KeyError: ['valor_errado']`. Confira `centro.columns` antes de informar o nome de uma coluna.

In [ ]:
import pandas as pd

centro = pd.read_csv("dataset/dre_filial_centro.csv")

try:
    centro.dropna(subset=["valor_errado"])
except KeyError as e:
    print(f"Erro: {e}")

### 🔹 Exemplo 4 — Tratando valores ausentes: dropna() x fillna()

📖 **Antes do código: `dropna()` e `fillna()`.** Os dois tratam valor ausente, mas de jeitos opostos. `dropna()` **remove a linha inteira** sempre que ela tiver pelo menos 1 valor ausente (em qualquer coluna, a não ser que você limite com `subset=[...]`, como viu no aviso de erro acima) — devolve um novo DataFrame, sem alterar o original. `fillna(valor)` faz o oposto: **mantém todas as linhas**, e substitui cada `NaN` pelo `valor` que você passar (aqui, a média da própria coluna, `centro["valor"].mean()`) — também devolve uma cópia nova, sem alterar o original. **Qual usar entre os dois?** `dropna()` quando a linha sem aquele dado não serve pra análise; `fillna()` quando faz mais sentido estimar um valor razoável no lugar do vazio.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")

sem_nulos = centro.dropna()              # remove a LINHA inteira que tem nulo
display(sem_nulos)
print(f"Shape depois do dropna(): {sem_nulos.shape}")

centro_com_fillna = centro.copy()
centro_com_fillna["valor"] = centro_com_fillna["valor"].fillna(centro_com_fillna["valor"].mean())
display(centro_com_fillna)

De 25 linhas, `centro.dropna()` derruba pra 20 — a linha inteira some se `valor` estiver vazio. Já `centro["valor"].fillna(centro["valor"].mean())` preenche cada vazio com `22277.50`, a média da própria coluna. Esse número ainda está longe de confiável por dois motivos que as próximas seções vão resolver: ele foi calculado **com a linha duplicada contando em dobro**, **com o outlier de `125000` puxando pra cima**, e ainda **misturando receita com despesa** numa única média — duas naturezas de valor bem diferentes.

> ⚠️ **Armadilha real:** `centro["valor"].fillna(0)` sozinho **não altera** `centro` — ele só devolve uma cópia com os nulos preenchidos. Para o DataFrame original mudar de fato, reatribua: `centro["valor"] = centro["valor"].fillna(0)`. Esquecer isso é o erro mais comum desta semana.

### ✏️ Atividade Prática 1 — Sua vez de programar

**Contextualização:** o financeiro do Restaurante Sabor Caseiro recebeu o DRE de Centro direto do sistema novo e precisa confirmar quantos valores estão faltando antes de fechar o mês.

**Comando:** usando o mesmo `dre_filial_centro.csv`, comece rodando `head()`, `tail()` e `info()` pra confirmar a estrutura da tabela (como no Exemplo 2). Depois, detecte os valores ausentes com `isna().sum()` e trate com `fillna()`, preenchendo com a **mediana** da coluna — o valor bem no meio, quando você ordena todos os números da coluna do menor pro maior; diferente da média, ela não é puxada por um valor muito alto ou muito baixo. Em pandas, isso é o método `.median()`, no lugar do `.mean()` do Exemplo 4 — reatribua o resultado. O valor da mediana é bem diferente do valor da média do Exemplo 4? Por quê (pense na linha duplicada, no outlier e na mistura de receita com despesa)?

In [ ]:
# (espaço para o código — construído ao vivo em aula)

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar, em uma frase, a diferença entre `dropna()` e `fillna()` — e por que `fillna()` sozinho, sem reatribuir, não muda o DataFrame original?

---

## 2. Duplicidades e Outliers

### 🔹 Exemplo 1 — Detectando e removendo duplicidades

📖 **Antes do código:** `duplicated()` compara cada linha da tabela com todas as anteriores e devolve uma Series de `True`/`False` — `True` só na 2ª (ou 3ª, 4ª...) ocorrência de uma linha idêntica; a primeira ocorrência sempre fica `False`, porque ainda não é repetição de nada. `drop_duplicates()` usa essa mesma lógica por trás, mas já devolve um novo DataFrame sem as linhas marcadas como repetidas — mantendo sempre a primeira ocorrência de cada uma.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")

print(centro.duplicated())          # True na 2ª ocorrência de uma linha idêntica

centro_sem_duplicatas = centro.drop_duplicates()
display(centro_sem_duplicatas)
print(centro_sem_duplicatas.shape)

Aqui, `duplicated()` marca `True` só no índice 24 (`Receita de Vendas`, Janeiro, repetido do índice 0). `drop_duplicates()` devolve uma cópia sem essa linha: de 25 linhas, sobram 24. Repare que isso não mexeu nos valores ausentes (continuam lá) — duplicidade e valor ausente são dois problemas independentes, cada um resolvido com uma técnica diferente.

> ⚠️ **Veja como é um erro real do Python.** Um erro de digitação comum é esquecer o "s" do plural: `centro.drop_duplicate()` — você vai ver `AttributeError: 'DataFrame' object has no attribute 'drop_duplicate'`. O método certo é `drop_duplicates()`, sempre no plural.

In [ ]:
import pandas as pd

centro = pd.read_csv("dataset/dre_filial_centro.csv")

try:
    centro.drop_duplicate()
except AttributeError as e:
    print(f"Erro: {e}")

### 🔹 Exemplo 2 — Detectando outliers com describe() e IQR

📖 **Antes do código:** `Receita de Vendas` e as categorias de despesa têm naturezas bem diferentes — uma é dinheiro que entra, as outras são dinheiro que sai, em escalas bem diferentes (a receita do mês é sempre maior que qualquer despesa isolada). Comparar os dois numa única conta de outlier não faz sentido: a receita pareceria "outlier" das despesas, sem ser. Por isso, a partir daqui, o `describe()`/IQR roda só nas linhas de **despesa**.

In [ ]:
import pandas as pd

centro = pd.read_csv("dataset/dre_filial_centro.csv")
centro_sem_duplicatas = centro.drop_duplicates()

despesas = centro_sem_duplicatas[centro_sem_duplicatas["tipo"] == "Despesa"]
print(despesas["valor"].describe())

Filtrando só as 20 linhas de despesa (das 24 sem duplicata), o `std` (desvio padrão) fica bem maior que a `mean` — sinal forte de que existe algum valor bem fora do padrão. O `max` confirma: nenhuma outra despesa do relatório chega perto desse valor.

📖 **Antes do código: o que são quartis e o que é IQR?** Quando você ordena todos os valores de uma coluna do menor pro maior, os **quartis** são os 3 pontos que dividem essa lista em 4 partes iguais (25% dos dados em cada parte):

- **Q1** (1º quartil) é o valor abaixo do qual ficam os 25% menores valores.
- **Q2** é a **mediana** — o valor bem no meio, com 50% dos dados abaixo e 50% acima. É a mesma "mediana" que você já usou na Atividade Prática 1, e é exatamente a linha `50%` que `describe()` mostra.
- **Q3** (3º quartil) é o valor abaixo do qual ficam os 75% menores valores (ou seja, os 25% maiores ficam acima dele).

Em pandas, `.quantile(0.25)` calcula Q1 e `.quantile(0.75)` calcula Q3 diretamente — é exatamente isso que o código a seguir faz.

O **IQR** ("Interquartile Range", amplitude interquartil) é `Q3 - Q1`: a distância entre o quartil de baixo e o de cima, ou seja, o intervalo onde ficam os 50% "do meio" dos dados — nem os valores mais baixos, nem os mais altos. Quanto maior o IQR, mais espalhados estão os valores centrais.

**Por que multiplicar o IQR por 1,5?** É uma regra prática amplamente usada em análise de dados (conhecida como regra de Tukey): qualquer valor que fique mais de 1,5 vez o IQR acima de Q3 (ou abaixo de Q1) é considerado estatisticamente distante o suficiente do resto pra ser tratado como outlier. O `1,5` não é uma lei fixa, é uma margem generosa, testada e adotada como padrão de mercado — um número menor, como `1,0`, marcaria outlier com mais frequência (mais falsos positivos); um número maior, como `3,0`, só pegaria valores muito mais extremos. Para o volume e o tipo de dado financeiro desta semana, `1,5` é o padrão que você vai usar.

In [ ]:
import pandas as pd

centro = pd.read_csv("dataset/dre_filial_centro.csv")
centro_sem_duplicatas = centro.drop_duplicates()
despesas = centro_sem_duplicatas[centro_sem_duplicatas["tipo"] == "Despesa"]

q1 = despesas["valor"].quantile(0.25)
q3 = despesas["valor"].quantile(0.75)
iqr = q3 - q1
limite_superior = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}, limite superior={limite_superior}")
print(despesas[despesas["valor"] > limite_superior])

`Q1` (`3150.0`) e `Q3` (`12300.0`) marcam onde ficam os 25% menores e os 25% maiores valores de despesa; `IQR` (`9150.0`) mede a dispersão típica entre eles. O limite superior (`Q3 + 1,5 × IQR = 26025.0`) é o ponto a partir do qual uma despesa é considerada outlier — e só `Despesas com Pessoal` de Abril, o `125000.0`, ultrapassa esse limite. Esse valor quase certamente é um erro de digitação: um zero a mais no lugar dos `12500` que a folha de pagamento dessa filial costuma custar.

💡 **Ordem importa:** se você calcular a média de `valor` (Exemplo 4 da Seção 1) *antes* de tratar esse outlier — e antes de separar despesa de receita — a média vem bem distorcida. Trate duplicidade e outlier, e separe receita de despesa, *antes* de usar qualquer média/desvio padrão para preencher nulos ou tomar decisões.

### 🔹 Exemplo extra — Um cuidado real ao remover o outlier

📖 **Antes do código:** parece natural filtrar assim: `despesas[despesas["valor"] <= limite_superior]` — "mantenha só quem está dentro do limite". Só que isso esconde uma armadilha quando a coluna tem valor ausente. A correção usa `~`, que inverte `True` e `False` numa coluna inteira de uma vez — é o mesmo `not` que você já usa desde a Semana 02, só que aplicado a um `Series` inteiro (todas as linhas de uma vez), não a um valor só.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")
centro_sem_duplicatas = centro.drop_duplicates()
despesas = centro_sem_duplicatas[centro_sem_duplicatas["tipo"] == "Despesa"]

q1 = despesas["valor"].quantile(0.25)
q3 = despesas["valor"].quantile(0.75)
limite_superior = q3 + 1.5 * (q3 - q1)

filtro_errado = despesas[despesas["valor"] <= limite_superior]
filtro_certo = despesas[~(despesas["valor"] > limite_superior)]

print(f"Filtro <= (errado): {filtro_errado.shape} — perdeu linhas com valor ausente")
print(f"Filtro ~ > (certo): {filtro_certo.shape} — manteve as linhas com valor ausente")
display(filtro_certo)

O filtro `<= limite_superior` devolve `(14, 4)`: além de remover `Despesas com Pessoal` de Abril (o outlier de verdade), ele **também removeu as 5 linhas com `valor` ausente**. Isso acontece porque, em pandas, qualquer comparação (`>`, `<`, `<=`, `>=`) com `NaN` sempre devolve `False` — então `NaN <= limite_superior` é `False`, e a linha some do resultado sem nenhum aviso. O filtro `~(valor > limite_superior)` devolve `(19, 4)`: ele nega a condição de outlier, e como `NaN > limite_superior` também é `False`, a negação `~False` é `True` — as 5 linhas com valor ausente ficam. **Regra prática:** ao remover outlier por comparação numérica, use a negação da condição de outlier (`~(coluna > limite)`), nunca a condição "oposta" escrita à mão (`coluna <= limite`) — os dois só são equivalentes quando não existe nenhum valor ausente na coluna.

### ✏️ Atividade Prática 2 — Sua vez de programar

**Contextualização:** o contador do Restaurante Sabor Caseiro vai usar o DRE de Centro pra fechar o resultado do mês, mas antes precisa confirmar, sem depender do seu olho, que só existe mesmo 1 outlier no relatório.

**Comando:** repita a técnica completa desta seção, sozinho e sem olhar os exemplos: `duplicated()` + `drop_duplicates()`, filtro por `tipo == "Despesa"`, seguido de Q1/Q3/IQR/limite superior. Confirme que chega nos mesmos números (24 linhas após remover a duplicata, outlier só em `Despesas com Pessoal` de Abril).

In [ ]:
# (espaço para o código — construído ao vivo em aula)

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar por que faz sentido separar receita de despesa antes de calcular o limite de outlier — e por que `coluna <= limite` e `~(coluna > limite)` podem devolver resultados diferentes quando a coluna tem valores ausentes?

---

## 3. Normalização e Colunas Calculadas/Condicionais

### 🔹 Exemplo 1 — Normalização min-max

📖 **Antes do código:** esta célula refaz, em ordem, o que as Seções 1 e 2 ensinaram (remove duplicata, separa despesa, remove outlier preservando valor ausente) pra só então calcular uma média confiável — mas repare no `.copy()` depois de cada filtro. `dados[dados["tipo"] == "Despesa"]` devolve, tecnicamente, uma "fatia" do DataFrame original, e tentar alterar essa fatia diretamente (como a linha `despesas["valor"] = ...` faz mais adiante) pode disparar um aviso do pandas (`SettingWithCopyWarning`) avisando que não está claro se você queria alterar a fatia ou o original. `.copy()` resolve isso na raiz: cria uma tabela nova e independente, então qualquer alteração em `despesas` daqui pra frente é inequivocamente só em `despesas`. É um hábito comum sempre que você filtra um DataFrame e pretende **modificar** o resultado do filtro em seguida.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")
dados = centro.drop_duplicates()
despesas = dados[dados["tipo"] == "Despesa"].copy()

q1 = despesas["valor"].quantile(0.25)
q3 = despesas["valor"].quantile(0.75)
limite_superior = q3 + 1.5 * (q3 - q1)
despesas = despesas[~(despesas["valor"] > limite_superior)].copy()

media_confiavel = despesas["valor"].mean()
despesas["valor"] = despesas["valor"].fillna(media_confiavel)

print(f"Média usada no fillna (só despesas, já sem outlier/duplicata): {media_confiavel:.2f}")
display(despesas)

O resultado: uma média de `6539.29` — bem menor que a `22277.50` distorcida do Exemplo 4 da Seção 1. Essa é a média **confiável**, porque já não tem outlier, duplicata nem receita misturada puxando o número pra um lugar errado.

📖 **Antes do código: a fórmula min-max.** Pra reescalar uma coluna pro intervalo de 0 a 1 sem perder a proporção entre os valores, a fórmula é `(valor - mínimo) / (máximo - mínimo)`: o menor valor da coluna sempre vira `0.0` (porque `mínimo - mínimo = 0`), o maior sempre vira `1.0` (porque `máximo - máximo`, dividido por si mesmo, dá `1`), e os valores do meio ficam proporcionalmente entre os dois extremos.

In [ ]:
import pandas as pd
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")
dados = centro.drop_duplicates()
despesas = dados[dados["tipo"] == "Despesa"].copy()
q1 = despesas["valor"].quantile(0.25)
q3 = despesas["valor"].quantile(0.75)
limite_superior = q3 + 1.5 * (q3 - q1)
despesas = despesas[~(despesas["valor"] > limite_superior)].copy()
despesas["valor"] = despesas["valor"].fillna(despesas["valor"].mean())

despesas["valor_normalizado"] = (
    (despesas["valor"] - despesas["valor"].min())
    / (despesas["valor"].max() - despesas["valor"].min())
)
display(despesas)

Normalizar é útil quando você precisa comparar categorias de despesa em uma mesma escala, sem que a categoria com o maior número em reais "domine" qualquer cálculo que combine as duas.

### 🔹 Exemplo 2 — Coluna calculada com transformação condicional

📖 **Antes do código:** `np.where(condição, valor_se_true, valor_se_false)` cria uma coluna nova numa linha só: pra cada linha da tabela, testa a `condição` e devolve `valor_se_true` quando ela é verdadeira, ou `valor_se_false` quando é falsa — sem precisar de um `for` percorrendo linha a linha nem de um `if` dentro dele. É a mesma ideia de operação vetorizada, sem `for`, que você já viu com arrays NumPy na Semana 05.

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

centro = pd.read_csv("dataset/dre_filial_centro.csv")
dados = centro.drop_duplicates()
despesas = dados[dados["tipo"] == "Despesa"].copy()
q1 = despesas["valor"].quantile(0.25)
q3 = despesas["valor"].quantile(0.75)
limite_superior = q3 + 1.5 * (q3 - q1)
despesas = despesas[~(despesas["valor"] > limite_superior)].copy()
despesas["valor"] = despesas["valor"].fillna(despesas["valor"].mean())

despesas["classificacao"] = np.where(despesas["valor"] > 10000, "Alto Impacto", "Baixo Impacto")
display(despesas)

Aqui, onde `valor > 10000`, o resultado é `"Alto Impacto"` (algumas linhas de `CMV` e `Despesas com Pessoal`); o resto vira `"Baixo Impacto"`.

### ✏️ Atividade Prática 3 — Sua vez de programar

**Contextualização:** a diretoria do Restaurante Sabor Caseiro quer identificar quais despesas têm o maior peso relativo no DRE de Centro, numa escala comparável (não no valor bruto em reais) — pra decidir onde vale mais a pena investir tempo de auditoria mês a mês.

**Comando:** usando as `despesas` já limpas (duplicata e outlier tratados, nulos preenchidos), normalize a coluna `valor` (min-max, como no Exemplo 1) e crie uma coluna `peso_relativo`, classificando com `np.where()` como `"Peso Alto"` quando `valor_normalizado > 0.5` e `"Peso Baixo"` caso contrário. O resultado é diferente do `classificacao` do Exemplo 2 (baseado no valor bruto)? Por quê?

In [ ]:
# (espaço para o código — construído ao vivo em aula)

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar por que `np.where()` é preferível a um `for` com `if` dentro para criar uma coluna condicional?

---

## 4. Concatenação e Pipeline Completo

### 🔹 A segunda filial: Zona Sul, ainda sem limpeza nenhuma

In [ ]:
import pandas as pd
from IPython.display import display

zona_sul = pd.read_csv("dataset/dre_filial_zona_sul.csv")
display(zona_sul)
print(zona_sul.shape)

Mesma estrutura do Centro (25 linhas, 6 categorias, 4 meses), mas com seus próprios problemas: 4 valores ausentes, 1 linha duplicada (`Receita de Vendas`, Fevereiro) e 1 outlier (`Despesas com Pessoal` de Fevereiro, em `105000`). Antes de juntar as duas filiais, cada uma precisa passar pela mesma limpeza que o Centro já passou.

📖 **Antes do código: o que `pd.concat()` faz?** `pd.concat([tabela1, tabela2], ...)` empilha as linhas de duas (ou mais) tabelas que têm as mesmas colunas, uma embaixo da outra, formando uma única tabela maior — basicamente "colar" um DataFrame debaixo do outro. Aqui, depois de limpar as despesas da Zona Sul, `pd.concat([receitas, despesas], ignore_index=True)` reúne de volta as receitas (que nunca precisaram de limpeza) com as despesas já tratadas, reconstruindo a tabela completa da filial. `ignore_index=True` refaz a numeração das linhas de 0 em diante, em vez de manter os índices originais espalhados de cada pedaço.

In [ ]:
import pandas as pd
from IPython.display import display

zona_sul = pd.read_csv("dataset/dre_filial_zona_sul.csv")
dados = zona_sul.drop_duplicates()
receitas = dados[dados["tipo"] == "Receita"]
despesas = dados[dados["tipo"] == "Despesa"].copy()

q1 = despesas["valor"].quantile(0.25)
q3 = despesas["valor"].quantile(0.75)
limite_superior = q3 + 1.5 * (q3 - q1)
despesas = despesas[~(despesas["valor"] > limite_superior)].copy()
despesas["valor"] = despesas["valor"].fillna(despesas["valor"].mean())

zona_sul_limpo = pd.concat([receitas, despesas], ignore_index=True)
print(f"Shape: {zona_sul_limpo.shape}")
display(zona_sul_limpo)

Esse é o mesmo pipeline da Seção 3, aplicado à Zona Sul: remove duplicata, separa receita de despesa, remove o outlier preservando valor ausente, preenche o nulo com a média confiável das despesas — e por fim usa `pd.concat([receitas, despesas], ignore_index=True)` pra juntar as receitas (que nunca precisaram de limpeza) de volta com as despesas já limpas, formando o `zona_sul_limpo` com 23 linhas (24 depois da duplicata − 1 outlier removido).

### 🔹 Combinando as duas filiais com pd.concat()

📖 **Antes do código: por que juntar as duas filiais?** Até agora, Centro e Zona Sul são 2 tabelas separadas — cada uma mostra só o resultado de uma filial. Só que a diretoria do Restaurante Sabor Caseiro não toma decisão olhando uma filial de cada vez: ela quer saber, por exemplo, "quanto a empresa INTEIRA gastou com CMV neste período?" — uma pergunta que nenhuma das duas tabelas sozinha responde, porque cada uma só tem a metade do dado. Você já viu `pd.concat()` empilhando receita e despesa de uma única filial (mesma técnica); agora a ideia é a mesma, só que empilhando as duas filiais inteiras já limpas — é esse DataFrame consolidado que faz sentido alimentar, por exemplo, um dashboard de Power BI.

In [ ]:
import pandas as pd
from IPython.display import display

# Centro, já limpo (mesmo pipeline)
centro = pd.read_csv("dataset/dre_filial_centro.csv")
dados_c = centro.drop_duplicates()
receitas_c = dados_c[dados_c["tipo"] == "Receita"]
despesas_c = dados_c[dados_c["tipo"] == "Despesa"].copy()
q1c = despesas_c["valor"].quantile(0.25)
q3c = despesas_c["valor"].quantile(0.75)
limite_c = q3c + 1.5 * (q3c - q1c)
despesas_c = despesas_c[~(despesas_c["valor"] > limite_c)].copy()
despesas_c["valor"] = despesas_c["valor"].fillna(despesas_c["valor"].mean())
centro_limpo = pd.concat([receitas_c, despesas_c], ignore_index=True)

# Zona Sul, já limpo (mesmo pipeline)
zona_sul = pd.read_csv("dataset/dre_filial_zona_sul.csv")
dados_z = zona_sul.drop_duplicates()
receitas_z = dados_z[dados_z["tipo"] == "Receita"]
despesas_z = dados_z[dados_z["tipo"] == "Despesa"].copy()
q1z = despesas_z["valor"].quantile(0.25)
q3z = despesas_z["valor"].quantile(0.75)
limite_z = q3z + 1.5 * (q3z - q1z)
despesas_z = despesas_z[~(despesas_z["valor"] > limite_z)].copy()
despesas_z["valor"] = despesas_z["valor"].fillna(despesas_z["valor"].mean())
zona_sul_limpo = pd.concat([receitas_z, despesas_z], ignore_index=True)

# Consolidando as 2 filiais
dre_consolidado = pd.concat([centro_limpo, zona_sul_limpo], ignore_index=True)
print(f"Shape do consolidado: {dre_consolidado.shape}")
display(dre_consolidado)
print(dre_consolidado.groupby("categoria")["valor"].sum().sort_values(ascending=False))

`pd.concat([centro_limpo, zona_sul_limpo], ignore_index=True)` empilha as linhas das duas filiais já limpas em um único DataFrame: `23 + 23 = 46` linhas. `ignore_index=True` refaz a numeração do índice de 0 a 45, em vez de repetir os índices originais de cada DataFrame. O `groupby("categoria")["valor"].sum()` no final mostra, pela primeira vez, o resultado **consolidado das 2 filiais juntas** — a visão que a diretoria do Restaurante Sabor Caseiro realmente precisa, e que nenhuma das duas filiais sozinha mostra.

Você acabou de ver a consolidação funcionando, escrita direto na célula. Agora vamos organizar essas mesmas etapas num formato reutilizável — a versão que rodaria de verdade todo mês, sem precisar reescrever nada à mão.

### Pipeline completo — juntando tudo em ordem, e exportando para o Power BI

📖 **A ordem das etapas não é arbitrária:**
1. Ler os arquivos reais (`pd.read_csv`) de cada filial
2. Remover duplicidades (`drop_duplicates`)
3. Separar receita de despesa, e remover outlier nas despesas, preservando valores ausentes (`~(coluna > limite)`)
4. Só então tratar os nulos restantes (`fillna`, com a média já confiável)
5. Reunir receita + despesa de cada filial, e consolidar as 2 filiais (`pd.concat`)
6. Exportar o resultado limpo para um arquivo novo — é esse arquivo que o time de BI vai conectar no Power BI

📖 **Antes do código, sobre o passo 6:** `dataframe.to_csv("nome_do_arquivo.csv", index=False)` grava um DataFrame inteiro num arquivo `.csv` novo — `index=False` evita gravar uma coluna extra só com o número da linha, que ninguém pediu. Se o arquivo já existir, ele é sobrescrito.

In [ ]:
import pandas as pd
from IPython.display import display

def limpar_filial(caminho_arquivo):
    dados = pd.read_csv(caminho_arquivo).drop_duplicates()
    receitas = dados[dados["tipo"] == "Receita"]
    despesas = dados[dados["tipo"] == "Despesa"].copy()

    q1 = despesas["valor"].quantile(0.25)
    q3 = despesas["valor"].quantile(0.75)
    limite_superior = q3 + 1.5 * (q3 - q1)
    despesas = despesas[~(despesas["valor"] > limite_superior)].copy()
    despesas["valor"] = despesas["valor"].fillna(despesas["valor"].mean())

    return pd.concat([receitas, despesas], ignore_index=True)

# 1-4: ler e limpar cada filial
centro_limpo = limpar_filial("dataset/dre_filial_centro.csv")
zona_sul_limpo = limpar_filial("dataset/dre_filial_zona_sul.csv")

# 5: consolidar as 2 filiais
dre_consolidado = pd.concat([centro_limpo, zona_sul_limpo], ignore_index=True)
print(f"Linhas consolidadas: {len(centro_limpo)} (Centro) + {len(zona_sul_limpo)} (Zona Sul) = {len(dre_consolidado)}")
display(dre_consolidado)

# 6: exportar para um arquivo novo, pronto para o Power BI
dre_consolidado.to_csv("dre_consolidado.csv", index=False)
print("Arquivo dre_consolidado.csv criado com sucesso.")

# Confirmação: reabre o arquivo exportado do zero, pra provar que ele existe e está correto
conferencia = pd.read_csv("dre_consolidado.csv")
display(conferencia)

A função `limpar_filial(caminho_arquivo)` empacota as etapas 2 a 4 (que antes você via espalhadas em várias células) num único bloco reutilizável — a mesma ideia de função que você já pratica desde a Semana 04, agora aplicada a um pipeline de limpeza inteiro. Chamá-la duas vezes, uma pra cada filial, evita repetir o mesmo código duas vezes. Esse `dre_consolidado.csv` é exatamente o tipo de arquivo que um time de BI conecta no Power BI (ou Tableau, ou qualquer outra ferramenta de dashboard): eles não tratam nulo, duplicata nem outlier lá — esperam receber o dado já limpo, pronto pra virar gráfico. A releitura do arquivo (`pd.read_csv("dre_consolidado.csv")`) no final da célula não é redundante — é a forma de confirmar que o arquivo realmente foi salvo do jeito esperado, sem depender só da mensagem de sucesso. É o mesmo princípio da Semana 07 que vem a seguir: um pipeline de dados de verdade termina entregando um arquivo confiável para a próxima etapa, não uma tela de código.

### ✏️ Atividade Prática 4 — Sua vez de programar

**Contextualização:** o Restaurante Sabor Caseiro vai automatizar esse pipeline pra rodar todo mês automaticamente, sem revisão manual, entregando o arquivo pronto pro time de BI — antes de aprovar isso, a diretoria pediu uma confirmação de que o pipeline completo funciona do início ao fim, incluindo a exportação.

**Comando:** rode o pipeline completo (as 6 etapas acima) numa única célula, do jeito que foi mostrado, confirme o `shape` final do `dre_consolidado`, e confirme que o arquivo `dre_consolidado.csv` foi criado (uma forma simples: `import os; print(os.path.exists("dre_consolidado.csv"))` — `os.path.exists(caminho)` devolve `True` se existir um arquivo naquele caminho, `False` se não existir). Depois, responda em uma frase: por que faz mais sentido o Python gerar esse arquivo pronto do que o time de BI receber os 2 arquivos sujos e ter que limpar tudo de novo dentro do Power BI?

In [ ]:
# (espaço para o código — construído ao vivo em aula)

---
## 5. Limpeza de Texto e Padronização

🔗 **Por que isso agora?** Tudo que você já fez nas Seções 1 a 4 foi limpeza de **número** (`NaN`, duplicata, outlier). Mas o dia a dia de quem trabalha com dados também lida o tempo inteiro com **texto sujo**: espaço sobrando na ponta de um nome, letra trocada por engano, palavra quebrada ao digitar, coluna de dinheiro com `"-"` no lugar de número. É exatamente o tipo de erro que aparece em qualquer planilha real — do RH, do financeiro, de cadastro de cliente — e as técnicas desta seção são a versão em **texto** do que você já aprendeu em **número**: os mesmos passos (detectar, decidir, corrigir, reatribuir), aplicados a outro tipo de dado.

### 📌 Um novo arquivo: cadastro de fornecedores do Restaurante Sabor Caseiro

O financeiro te passou mais um arquivo — `fornecedores_despesas.csv`, com os 14 fornecedores que geram as despesas do DRE. Ele chegou com os mesmos tipos de erro que você provavelmente já viu numa planilha real: espaço sobrando no nome, letra trocada por engano (`@` no lugar de `a`), palavra quebrada com hífen e vírgula sobrando, e uma coluna de valor com `"-"` em vez de número. **Rodando no Colab?** Envie (ou confirme que já tem na pasta) o arquivo `fornecedores_despesas.csv`, do mesmo jeito que você já fez com os arquivos da semana.

### 🔹 Exemplo 1 — Vendo os dados brutos, sem filtro nenhum

📖 **Antes do código:** assim como fez com o DRE na Seção 1, antes de corrigir qualquer coisa, vale ver os dados exatamente como chegaram.

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")
display(fornecedores)
print(fornecedores.dtypes)

Repare em pelo menos 4 problemas diferentes, mesmo sem rodar nenhum comando de limpeza ainda: nomes com espaço sobrando no início ou no fim (`" Distribuidora Boa Carne"`), um `@` no lugar de uma letra (`"Frigorífico S@o Jorge"`, `"Cont@bil"`), palavras quebradas com hífen e uma vírgula sobrando (`"Pre-ço Justo,"`, `"Sil-va,"`), e a coluna `valor_mensal` com `"-"` em algumas linhas. Por causa desse `"-"` misturado com números, `dtypes` mostra `valor_mensal` como texto (`object`), não como número — é o mesmo problema que você teria se tentasse somar essa coluna agora: o Python não soma texto com número.

### 🔹 Exemplo 2 — Espaço sobrando e caractere errado: `.str.strip()` e `.str.replace()`

📖 **Antes do código:** toda Series de texto no pandas tem um conjunto de métodos que começam com `.str` — é assim que você aplica uma operação de texto em **toda a coluna de uma vez**, sem `for`. `.str.strip()` remove espaços (e quebras de linha) do **início e do fim** do texto — não mexe nos espaços que estão no meio, só nas pontas. `.str.replace("@", "a")` troca toda ocorrência de um pedaço de texto por outro, em cada valor da coluna — aqui, troca qualquer `@` digitado por engano pela letra `a` de verdade, em qualquer posição do texto.

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip()
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.replace("@", "a")

display(fornecedores)

`.str.strip()` tira o espaço antes de `"Distribuidora Boa Carne"` e depois de `"Frigorífico S@o Jorge"`/`"Dedetizadora Limpa Tudo"` — sem isso, `" Distribuidora Boa Carne"` e `"Distribuidora Boa Carne"` seriam tratados como dois valores diferentes, mesmo parecendo iguais aos seus olhos. Em seguida, `.str.replace("@", "a")` corrige `"S@o Jorge"` → `"Sao Jorge"`, `"Cont@bil"` → `"Contabil"`, `"Cred@amigo"` → `"Credaamigo"` e `"Vigi@"` → `"Vigia"` — qualquer `@` que aparecer no texto vira `a`, de uma vez, em todas as linhas.

### 🔹 Exemplo 3 — Palavra quebrada com hífen e vírgula sobrando

📖 **Antes do código:** é comum alguém digitar uma palavra grande e, sem querer, quebrar ela ao meio com um hífen (`"dife-rentes"` em vez de `"diferentes"`) — às vezes ainda sobra uma vírgula colada no final. A correção é a mesma lógica do Exemplo 2: mais um `.str.replace()`, agora removendo o hífen e a vírgula (substituindo os dois por texto vazio `""`, ou seja, apagando).

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# repetindo a limpeza do Exemplo 2 (espaço nas pontas e caractere errado)
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip()
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.replace("@", "a")

# limpeza nova deste exemplo: hífen quebrando a palavra + vírgula sobrando
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.replace("-", "")
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.replace(",", "")

fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.strip()
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.replace("-", "")
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.replace(",", "")

display(fornecedores)

`"Pre-ço Justo,"` vira `"Preço Justo"`, `"Sil-va,"` vira `"Silva"`, `"Emb-alagens,"` vira `"Embalagens"` e `"Rapi-da,"` vira `"Rapida"` — o hífen que quebrava a palavra e a vírgula sobrando somem, sem sobrar espaço nenhum estranho no meio. A mesma limpeza foi aplicada em `categoria_despesa` (com `.str.strip()` primeiro, porque essa coluna também tinha espaço sobrando em algumas linhas), garantindo que as 3 categorias que aparecem nesse arquivo (`Despesas Administrativas`, `CMV`, `Despesas Financeiras`) fiquem sempre escritas do mesmo jeito — essencial pra qualquer `groupby()` funcionar direito daqui pra frente: um `"CMV"` e um `"CMV "` (com espaço) formariam dois grupos diferentes, mesmo sendo a mesma categoria.

### 🔹 Exemplo 4 — Coluna de dinheiro digitada com `"-"`: `pd.to_numeric()`

📖 **Antes do código:** é muito comum, numa planilha financeira, alguém digitar `"-"` numa célula de valor pra dizer "não teve nada aqui neste mês" — só que isso transforma a coluna inteira em texto, porque `"-"` não é um número. `pd.to_numeric(coluna, errors="coerce")` tenta converter cada valor da coluna pra número; quando não consegue (como no `"-"`), em vez de travar com erro, ela troca esse valor por `NaN` — o mesmo `NaN` que você já trata desde a Seção 1, com `isna()`/`fillna()`.

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# repetindo a limpeza de texto dos Exemplos 2 e 3
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip().str.replace("@", "a").str.replace("-", "").str.replace(",", "")
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.strip().str.replace("-", "").str.replace(",", "")

# limpeza nova deste exemplo: coluna de dinheiro com "-" no lugar de número
fornecedores["valor_mensal"] = pd.to_numeric(fornecedores["valor_mensal"], errors="coerce")
display(fornecedores)
print(f"Valores que viraram NaN: {fornecedores['valor_mensal'].isna().sum()}")

Os 3 `"-"` do arquivo (fornecedores que não geraram despesa naquele mês) viram `NaN`, e a coluna inteira passa a ser número (`float64`), pronta pra soma, média ou qualquer conta. Repare que a decisão de preencher esse `NaN` é diferente da que você tomou na Seção 1: lá, o `NaN` significava "não sei o valor real", e fazia sentido estimar com a média. Aqui, o `"-"` já significa "não teve despesa nenhuma" — então o valor certo pra preencher não é uma média, é `0`:

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# repetindo a limpeza dos Exemplos 2, 3 e 4 (texto + conversão pra número)
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip().str.replace("@", "a").str.replace("-", "").str.replace(",", "")
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.strip().str.replace("-", "").str.replace(",", "")
fornecedores["valor_mensal"] = pd.to_numeric(fornecedores["valor_mensal"], errors="coerce")

# limpeza nova deste exemplo: decide o que fazer com o NaN que veio do "-"
fornecedores["valor_mensal"] = fornecedores["valor_mensal"].fillna(0)
display(fornecedores)
print(f"Total gasto com todos os fornecedores: R$ {fornecedores['valor_mensal'].sum():.2f}")

O total (`R$ 24640.00`) já considera os 3 fornecedores que não geraram despesa naquele mês como `0`, não como um valor estimado — é essa a diferença entre "não sei quanto foi" (Seção 1, preenche com média) e "sei que foi zero" (aqui, preenche com `0`). Escolher entre os dois depende sempre do que o `"-"`/`NaN` significa **no contexto do dado**, nunca de uma regra fixa.

### ✏️ Atividade Prática 5 — Sua vez de programar

**Contextualização:** o financeiro do Restaurante Sabor Caseiro quer o total gasto por categoria de despesa (`Despesas Administrativas`, `CMV`, `Despesas Financeiras`) já com o cadastro de fornecedores limpo, pra comparar com o que o DRE consolidado (Seção 4) mostrou.

**Comando:** leia `fornecedores_despesas.csv` de novo numa célula nova, aplique a limpeza completa desta seção (`.str.strip()`, `.str.replace()` pro `@` e pro hífen/vírgula em `fornecedor` e em `categoria_despesa`, e `pd.to_numeric(...).fillna(0)` em `valor_mensal`), e calcule `groupby("categoria_despesa")["valor_mensal"].sum()`, ordenado do maior pro menor.

In [ ]:
# (espaço para o código — construído ao vivo em aula)

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar, em uma frase, a diferença entre `.str.strip()` e `.str.replace()` — e por que preencher o `"-"` com `0` faz mais sentido aqui do que preencher com a média?

---

### 🔹 Exemplo 5 — Padronizando o cabeçalho das colunas

📖 **Antes do código:** o cabeçalho de uma tabela (o nome das colunas) também pode precisar de padronização, principalmente quando você recebe arquivos de sistemas diferentes, cada um com sua própria convenção. `df.columns` é a lista com os nomes das colunas, e ela também aceita os métodos `.str`: `.str.upper()` deixa tudo maiúsculo, `.str.lower()` deixa tudo minúsculo, e `.str.title()` deixa a **primeira letra de cada palavra** maiúscula, útil pra nomes com `_`, como `categoria_despesa` → `Categoria_Despesa`. (Existe também `.str.capitalize()`, que deixa maiúscula só a primeira letra de todo o texto, mas na prática `.title()` é o que você vai usar quase sempre com nome de coluna.)

In [ ]:
import pandas as pd

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# esta célula olha só o NOME das colunas, não depende da limpeza de linha estar feita
print("Maiúsculo:", list(fornecedores.columns.str.upper()))
print("Minúsculo:", list(fornecedores.columns.str.lower()))
print("Título (cada palavra):", list(fornecedores.columns.str.title()))

Nenhum desses comandos altera `fornecedores.columns` sozinho, cada um só **devolve** uma nova lista de nomes; pra aplicar de verdade, você reatribui: `fornecedores.columns = fornecedores.columns.str.title()` (a mesma regra de sempre reatribuir que você já viu com `fillna()`, na Seção 1). Pra renomear **uma única coluna**, sem mexer nas outras, use `.rename(columns={...})`:

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# repetindo a limpeza completa dos exemplos anteriores
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip().str.replace("@", "a").str.replace("-", "").str.replace(",", "")
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.strip().str.replace("-", "").str.replace(",", "")
fornecedores["valor_mensal"] = pd.to_numeric(fornecedores["valor_mensal"], errors="coerce").fillna(0)

# operação nova desta célula: renomear só a coluna de valor
fornecedores = fornecedores.rename(columns={"valor_mensal": "valor_mensal_reais"})
display(fornecedores)

`.rename(columns={"nome_antigo": "nome_novo"})` troca só o nome que você especificar no dicionário, mantendo os outros exatamente como estavam — diferente de `.str.upper()`/`.str.title()`, que mexem em **todas** as colunas de uma vez. `.rename()` também **devolve** uma cópia por padrão, por isso a reatribuição (`fornecedores = fornecedores.rename(...)`).

### 🔹 Exemplo 6 — Substituindo um valor específico dentro de uma coluna

📖 **Antes do código:** existe uma diferença importante entre os dois `.replace()` que você já viu nesta semana. `.str.replace("@", "a")` (Exemplo 2) troca um **pedaço** de texto, em **qualquer posição**, dentro de cada valor — funciona mesmo que o `@` esteja no meio de uma palavra grande. Já `.replace("valor_antigo", "valor_novo")`, **sem** o `.str` na frente, troca o valor **inteiro**, só quando ele bate certinho (igual, letra por letra) com `"valor_antigo"` — é o comando certo pra corrigir ou expandir uma categoria inteira, como a sigla `"CMV"` (Custo da Mercadoria Vendida).

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# repetindo a limpeza completa dos exemplos anteriores
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip().str.replace("@", "a").str.replace("-", "").str.replace(",", "")
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.strip().str.replace("-", "").str.replace(",", "")
fornecedores["valor_mensal"] = pd.to_numeric(fornecedores["valor_mensal"], errors="coerce").fillna(0)

# operação nova desta célula: substituir a sigla pelo nome completo da categoria
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].replace("CMV", "Custo da Mercadoria Vendida")
display(fornecedores)

Toda linha que tinha exatamente `"CMV"` agora tem `"Custo da Mercadoria Vendida"`. Se você tivesse usado `.str.replace("CMV", "Custo da Mercadoria Vendida")` aqui, o resultado seria igual **neste caso específico** (porque nenhuma outra categoria contém a sequência `"CMV"` escondida no meio do texto) — mas os dois comandos não são a mesma coisa, e usar `.str.replace()` por hábito em vez de `.replace()` pode trocar um pedaço de texto sem querer, em vez do valor inteiro. **Regra prática:** valor inteiro conhecido (uma categoria, uma sigla, um nome específico) → `.replace()`; pedaço de texto que pode aparecer em qualquer posição (um caractere errado, um trecho repetido) → `.str.replace()`.

### ✏️ Atividade Prática 6 — Sua vez de programar

**Contextualização:** o financeiro pediu pra padronizar o arquivo de fornecedores antes de importar num sistema novo: o cabeçalho precisa seguir o padrão da empresa (primeira letra de cada palavra maiúscula), e a categoria `"Despesas Administrativas"` precisa ser renomeada pra `"Despesas Operacionais"` (novo nome oficial adotado pela contabilidade).

**Comando:** padronize o cabeçalho das colunas com `.str.title()` (reatribuindo em `fornecedores.columns`), e substitua o valor `"Despesas Administrativas"` por `"Despesas Operacionais"` na coluna de categoria, usando o comando certo (pense: é um valor inteiro conhecido, ou um pedaço de texto solto?).

In [ ]:
# (espaço para o código — construído ao vivo em aula)

---
✅ **Checagem rápida — antes de avançar:**
Você consegue explicar, em uma frase, quando usar `.replace()` e quando usar `.str.replace()`?

---

### 🔹 Exportando o cadastro de fornecedores limpo

📖 **Antes do código:** assim como você fez na Seção 4 com o DRE consolidado, o passo final de qualquer limpeza é entregar um **arquivo pronto** — não deixar o resultado preso só na memória do notebook, que se perde quando a sessão termina. `to_csv(...)` grava o `fornecedores` já limpo (sem espaço sobrando, sem `@`, sem hífen quebrando palavra, com `valor_mensal_reais` numérico de verdade) num arquivo novo.

In [ ]:
import pandas as pd
from IPython.display import display

fornecedores = pd.read_csv("dataset/fornecedores_despesas.csv")

# repetindo a limpeza completa desta seção (texto + valor monetário)
fornecedores["fornecedor"] = fornecedores["fornecedor"].str.strip().str.replace("@", "a").str.replace("-", "").str.replace(",", "")
fornecedores["categoria_despesa"] = fornecedores["categoria_despesa"].str.strip().str.replace("-", "").str.replace(",", "")
fornecedores["valor_mensal"] = pd.to_numeric(fornecedores["valor_mensal"], errors="coerce").fillna(0)

fornecedores.to_csv("fornecedores_despesas_limpo.csv", index=False)
print("Arquivo fornecedores_despesas_limpo.csv criado com sucesso.")

# Confirmação: reabre o arquivo exportado do zero, pra provar que ele existe e está correto
conferencia_fornecedores = pd.read_csv("fornecedores_despesas_limpo.csv")
display(conferencia_fornecedores)

`fornecedores_despesas_limpo.csv` é a mesma ideia de entrega da Seção 4: um dado que chegou sujo (nome com espaço/caractere errado, valor como texto) e agora sai como arquivo limpo, pronto pra qualquer ferramenta usar sem precisar tratar nada de novo. Reabrir o arquivo com `pd.read_csv(...)` logo depois de exportar não é redundante — é a forma de confirmar que ele realmente foi salvo do jeito esperado, sem depender só da mensagem de sucesso.

---
### 🏁 Fechamento — Semana 06

**Nesta semana você aprendeu:**
- Detectar e tratar valores ausentes (NaN) com `isna()`, `dropna()` e `fillna()` — e por que reatribuir importa
- Detectar e remover duplicidades com `duplicated()`/`drop_duplicates()`, e outliers com `describe()`/IQR — separando naturezas diferentes de dado e preservando valores ausentes no filtro
- Normalizar colunas (min-max) e criar colunas condicionais com `np.where()`
- Limpar texto sujo com `.str.strip()`/`.str.replace()`, converter coluna de dinheiro com `pd.to_numeric(errors="coerce")`, padronizar cabeçalho de coluna e substituir valores com `.replace()`
- Combinar DataFrames com `pd.concat()`, montar um pipeline de limpeza completo reutilizável, e **exportar o resultado para um arquivo novo, pronto para ferramentas como o Power BI**

**Próxima semana:** Semana 07 — Visualização e Pipelines: comunicar insights com gráficos e montar um pipeline de dados completo.

---
## 6. Treino em Squads — Sexta-feira (Encontro 3)

Esta seção é usada **em sala (ou em salas remotas/breakout)** na sexta-feira. Cada squad recebe um dataset real diferente, cada um com **um problema principal diferente** — a missão é decidir a estratégia certa para aquele problema, não aplicar a mesma receita nos 4.

**Antes de começar:** envie (ou confirme que já tem na pasta) só o arquivo do **seu** squad — não precisa dos outros 3.

### Squad B — Cadastro de funcionários (valores ausentes no salário)

**Contextualização:** o RH de uma empresa precisa fechar a folha de pagamento do mês, mas o relatório de salários chegou com informação faltando pra alguns funcionários.

**Comando:** comece com `head()` e `info()` pra conhecer a estrutura do dataset (mesma dupla de comandos da Seção 1). Depois, rode `isna().sum()`, decida entre `dropna()` e `fillna()` pra tratar os valores ausentes de `salario`, e justifique a escolha pra turma.

In [ ]:
import pandas as pd
dataset_squad_b = pd.read_csv("dataset/dataset_squad_b_funcionarios.csv")

# explore: isna(), dropna(), fillna() — qual decisão faz mais sentido aqui?


### Squad C — Pedidos de um outro restaurante (linhas duplicadas)

**Contextualização:** o sistema de pedidos de um restaurante diferente do Sabor Caseiro travou por alguns segundos e reenviou 2 pedidos duas vezes — o gerente precisa saber o faturamento real do dia, sem contar nada em dobro.

**Comando:** comece com `head()` e `info()` pra conhecer a estrutura do dataset. Depois, rode `duplicated()` e `drop_duplicates()`, e confirme quantos pedidos (e qual valor total) são de verdade.

In [ ]:
import pandas as pd
dataset_squad_c = pd.read_csv("dataset/dataset_squad_c_pedidos.csv")

# explore: duplicated(), drop_duplicates()


### Squad D — Leituras de um sensor de temperatura (outlier)

**Contextualização:** um sensor industrial registrou uma leitura completamente fora do padrão no meio do turno — antes de calcular a temperatura média do período, é preciso decidir o que fazer com essa leitura.

**Comando:** comece com `head()` e `info()` pra conhecer a estrutura do dataset. Depois, rode `describe()` e o método IQR pra confirmar qual leitura é o outlier, e decida (com justificativa) se ela deve ser removida ou tratada de outro jeito.

In [ ]:
import pandas as pd
dataset_squad_d = pd.read_csv("dataset/dataset_squad_d_sensor.csv")

# explore: describe(), IQR


### Squad E — Notas de turmas em escalas diferentes (precisa normalizar)

**Contextualização:** a coordenação pedagógica quer comparar o desempenho de duas turmas, mas uma foi avaliada numa escala de 0 a 10 e a outra numa escala de 0 a 100 — comparar os números brutos dá uma ideia errada de quem foi melhor.

**Comando:** comece com `head()` e `info()` pra conhecer a estrutura do dataset. Depois, normalize a coluna `nota` (min-max) e compare as duas turmas de forma justa.

> ⚠️ **Cuidado com uma armadilha real aqui:** se você normalizar `nota` usando o mínimo e o máximo de **todo o dataset** (as duas turmas juntas), o resultado não fica justo — o mínimo vem da turma A (escala 0-10) e o máximo vem da turma B (escala 0-100), então a turma A inteira fica espremida perto de 0 e a turma B perto de 1, não importa o quão bem cada aluno foi **dentro da própria turma**. O jeito certo é normalizar **turma por turma**: filtre as notas de cada turma separadamente (mesmo padrão de filtro que você já usou nas Seções 2 e 3, trocando `tipo == "Despesa"` por `turma == "A"`) e aplique a fórmula min-max (Seção 3, Exemplo 1) em cada filtro, separadamente:
> ```python
> turma_a = dataset_squad_e[dataset_squad_e["turma"] == "A"].copy()
> turma_b = dataset_squad_e[dataset_squad_e["turma"] == "B"].copy()
>
> turma_a["nota_normalizada"] = (turma_a["nota"] - turma_a["nota"].min()) / (turma_a["nota"].max() - turma_a["nota"].min())
> turma_b["nota_normalizada"] = (turma_b["nota"] - turma_b["nota"].min()) / (turma_b["nota"].max() - turma_b["nota"].min())
> ```
> Assim cada turma vira uma escala de 0 a 1 baseada só nela mesma, e aí sim dá pra comparar o desempenho relativo de cada uma.

In [ ]:
import pandas as pd
dataset_squad_e = pd.read_csv("dataset/dataset_squad_e_notas.csv")

# explore: normalização min-max — como comparar as notas das 2 turmas de forma justa?


### 🗣️ Debate coletivo (após as apresentações)

Depois que todos os squads apresentarem, discuta com a turma:

- Por que a mesma técnica (ex.: `dropna()`) não era a melhor escolha para todos os datasets?
- O que aconteceria se o Squad D tivesse usado `fillna()` com a média sem antes tratar o outlier?
- O que aconteceria se cada squad tivesse aplicado a solução de outro squad no seu próprio dataset?

### Desafio (opcional)

Combine os datasets dos Squads B e C com `pd.concat()` (apenas as colunas em comum) e trate os problemas de ambos no resultado combinado.

---
### Assinatura

Curso: **Análise de Dados com Python — SENAI (Turma T5)**
Semana 06 — Limpeza e Transformação de Dados

*Prof. Especialista Cláudio F. Neves*